### **Day 5: Structural and Behavioral Patterns**

#### **Module 6: Structural Patterns**

* **Facade:** Simplifying complex internal subsystems for external consumers.  
* **Flyweight:** Using `__get__`, `__set__` and Factory pools to manage massive object counts.  

#### **Module 7: Behavioral Patterns**
* **Chain of Responsibility**: Delegate handling of responsibility by a chain of handlers.
* **Command**: Encapsulate a request as an object, so you can queue it, log it, and reverse it — without the caller knowing what it does.
* **Mediator**: Reduce chaotic dependencies between objects.
* **Strategy**: Defining a family of algorithms, enabling registration and selection in runtime.
* **Template**: Define an interface and allow alrogithms to be delegated to implementations of that interface.
* **Visitor**: Seperate algorithms from objects on which they operate.
* **Observer:** Implementing Pub/Sub systems using Callbacks and Event objects.  
* **State:** Replacing massive if/elif blocks with polymorphic State classes.  
* **Memento:** Capturing and restoring object internal state (Undo/Redo mechanisms).

#### **Odds and Ends**
* Pattern selection guide - a cheatsheet
* **Capstone Project**: Refactor a dirty codebase employing design patterns for clean design and implementation.

---

#### Facade Pattern

Provide a simplified interface using a scaffolding function / methods of a object that takes care intricacies of initializing
and working with inner composite objects.


In [ ]:
class Lights:
 
    def on(self):
        print("Turning on the lights.")

    def off(self):
        print("Turning off the lights.")

    def set_brightness(self, level):
        print(f"Setting brightness to {level}.")

class Fan:
    def on(self):
        print("Turning on the fan.")

    def off(self):
        print("Turning off the fan.")
    
    def set_speed(self, speed):
        print(f"Setting fan speed to {speed}.")

class Room:
    def __init__(self):
        self.lights = Lights()
        self.fan = Fan()
   
    def enter(self):
        self.lights.on()
        self.fan.on()

    def exit(self):
        self.lights.off()
        self.fan.off()

    def dim(self):
        self.lights.set_brightness(30)

    def brighten(self):
        self.lights.set_brightness(100)

    def cool(self):
        self.fan.set_speed(5)

    def warm(self):
        self.fan.set_speed(1)

room = Room()
room.enter()
room.dim()
room.cool()
room.brighten()
room.exit()

Turning on the lights.
Turning on the fan.
Setting brightness to 30.
Setting fan speed to 5.
Setting brightness to 100.
Turning off the lights.
Turning off the fan.


In [15]:
from facade_example import HomeTheaterFacade

theater = HomeTheaterFacade()
theater.watch_movie("Inception")
theater.end_movie()


--- Getting ready for the movie! ---
Lights: Dimming for the show...
Projector: Warming up...
Projector: Setting input to Streaming Box
Amp: Powering on...
Amp: Volume set to 15
App: Authenticating user...
App: Now streaming 'Inception'
--- Enjoy your film! ---


--- Shutting down theater... ---
App: Logging out.
Amp: Shutting down...
Projector: Cooling down...
Lights: Turning back up.
Theater is off.


##### Facade pattern examples in Python standard library

In [ ]:
import logging

# Facade — one line, just works
logging.warning("Something went wrong")
logging.info("App started")

# What the Facade is hiding underneath:
logger   = logging.getLogger("root")       # Logger object
handler  = logging.StreamHandler()         # Handler
formatter = logging.Formatter('%(levelname)s:%(name)s:%(message)s')
handler.setFormatter(formatter)            # attach formatter
logger.addHandler(handler)                 # attach handler
logger.setLevel(logging.WARNING)

logger.warning("Something went wrong")     # now call it

In [ ]:
from pathlib import Path

p = Path("/tmp/data")

# Each of these is a Facade call hiding the module it delegates to:
p.mkdir(parents=True, exist_ok=True)    # os.makedirs()
p.exists()                              # os.path.exists()
p.stat().st_size                        # os.stat()
p.read_text()                           # open() + .read() + .close()
p.write_text("hello")                   # open() + .write() + .close()
(p / "sub" / "file.txt").resolve()      # os.path.join() + os.path.abspath()

for f in p.glob("**/*.py"):             # os.walk() + fnmatch
    print(f)

In [14]:
import urllib.request

# Facade — one call
response = urllib.request.urlopen("https://httpbin.org/get")
print(response.read().decode())

# What urlopen() hides:
opener  = urllib.request.OpenerDirector()
opener.add_handler(urllib.request.HTTPHandler())
opener.add_handler(urllib.request.HTTPSHandler())
opener.add_handler(urllib.request.HTTPCookieProcessor())
opener.add_handler(urllib.request.HTTPRedirectHandler())
opener.add_handler(urllib.request.UnknownHandler())
request = urllib.request.Request("https://httpbin.org/get")
response = opener.open(request)         # now actually send
print(response.read().decode())

{
  "args": {}, 
  "headers": {
    "Accept-Encoding": "identity", 
    "Host": "httpbin.org", 
    "User-Agent": "Python-urllib/3.12", 
    "X-Amzn-Trace-Id": "Root=1-6a04b039-36db333641a165871098cc96"
  }, 
  "origin": "49.204.143.183", 
  "url": "https://httpbin.org/get"
}

{
  "args": {}, 
  "headers": {
    "Accept-Encoding": "identity", 
    "Host": "httpbin.org", 
    "User-Agent": "Python-urllib/3.12", 
    "X-Amzn-Trace-Id": "Root=1-6a04b03a-4b8e02f719b0f4b727bc9e05"
  }, 
  "origin": "49.204.143.183", 
  "url": "https://httpbin.org/get"
}



In [18]:
import subprocess

# Facade — clean and simple
result = subprocess.run(
    ["uname", "-a"],
    capture_output=True,
    text=True
)
print(result.stdout)

# What run() hides — the raw Popen machinery:
proc = subprocess.Popen(
    ["uname", "-a"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
stdout, stderr = proc.communicate()     # handle blocking I/O
proc.wait()                             # wait for exit
result = stdout.decode()
print(result)

Darwin Telesto.local 25.4.0 Darwin Kernel Version 25.4.0: Thu Mar 19 19:33:25 PDT 2026; root:xnu-12377.101.15~1/RELEASE_ARM64_T6041 arm64

Darwin Telesto.local 25.4.0 Darwin Kernel Version 25.4.0: Thu Mar 19 19:33:25 PDT 2026; root:xnu-12377.101.15~1/RELEASE_ARM64_T6041 arm64



---

#### Flyweight Pattern

Flywieght Patterns are used to consolidate and underlying optimize memory usage of attributes of multiple instances of a class using a federated (consolidated) backend implementation.

In [ ]:
class String:
    def __init__(self, max_length):
        self.max_length = max_length
        self.values = {}

    def __get__(self, instance, owner=None):
        _id = id(instance)
        if _id not in self.values:
            raise AttributeError("value not set")
        return self.values[_id]
    
    def __set__(self, instance, value):
        _id = id(instance)
        if type(value) is not str:
            raise TypeError("value must be a string")
        if len(value) > self.max_length:
            raise ValueError(f"value exceeds maximum length of {self.max_length}")
        self.values[_id] = value

class Number:
    def __init__(self, range):
        self.range = range
        self.values = {}

    def __get__(self, instance, owner=None):
        _id = id(instance)
        if _id not in self.values:
            raise AttributeError("value not set")
        return self.values[_id]
    
    def __set__(self, instance, value):
        _id = id(instance)
        if type(value) is not int:
            raise TypeError("value must be a number")
        if value not in self.range:
            raise ValueError(f"value {value} out of range {self.range}")
        self.values[_id] = value

class User:
    name = String(max_length=15)
    age = Number(range(0, 150))

    def __init__(self, name=None, age=None):
        if name is not None:
            self.name = name
        if age is not None:
            self.age = age
        
    def __repr__(self):
        return f"User(name='{self.name}', age={self.age})"
    

u1 = User()
u1.name = "Chandra"
u1.age = 30
print(u1)

u2 = User(name="Alice", age=25)
print(u2)

u3 = User(name="Bob", age=30)
print(u3)

print(User.__dict__['name'].values)

User(name='Chandra', age=30)
User(name='Alice', age=25)
User(name='Bob', age=30)
{5705833200: 'Chandra', 5705840592: 'Alice', 5705834064: 'Bob'}


In [ ]:
from user_flyweight import User

u1 = User("Alice")
u1.role = "admin"

u2 = User("Bob")
u2.role = "user"

u3 = User("Charlie")
u3.role = "user"

print(u1, u2, u3, sep="\n")

User({'name': 'Alice', 'role': 'admin'})
User({'name': 'Bob', 'role': 'user'})
User({'name': 'Charlie', 'role': 'user'})


##### Flyweight patterns in the Python standard library

In [ ]:
# Small Integer Caching — The most fundamental Flyweight in Python
# Python pre-creates and reuses integer objects in the range -5 to 256 at interpreter startup.

import sys

# Flyweight — same object reused, not recreated
a = 100
b = 100
print(a is b)           # True — exact same object in memory
print(id(a) == id(b))   # True

# Outside the cache range — new objects created each time
x = 1000
y = 1000
print(x is y)           # False — different objects (CPython)
print(id(x) == id(y))   # False

# The cache saves enormous memory in real programs
# where small integers are used millions of times
nums = [1, 1, 1, 1, 1]
print(all(n is nums[0] for n in nums))   # True — all point to same object!

# Verify with sys.getrefcount
print(sys.getrefcount(1))    # very high — shared by entire runtime
print(sys.getrefcount(1000)) # low — freshly created

In [ ]:
# String Interning — Flyweight for string literals
# Python automatically interns string literals that look like identifiers,
# and you can force interning via sys.intern().

import sys

# Auto-interned — compile-time string literals
a = "hello"
b = "hello"
print(a is b)           # True — same interned object

# Strings that look like identifiers are auto-interned
s1 = "python_rocks"
s2 = "python_rocks"
print(s1 is s2)         # True (CPython)

# Strings with spaces are NOT auto-interned
s3 = "hello world"
s4 = "hello world"
print(s3 is s4)         # False — separate objects

# Force interning with sys.intern() — explicit Flyweight
s5 = sys.intern("hello world")
s6 = sys.intern("hello world")
print(s5 is s6)         # True — now the same object

# Real-world benefit: dictionary key lookups become pointer comparisons
# instead of character-by-character string comparisons
large_dict = {sys.intern(f"key_{i}"): i for i in range(10000)}

# This lookup is O(1) pointer comparison, not O(n) string comparison
key = sys.intern("key_5000")
print(large_dict[key])   # 5000

#### Memoize Pattern

Provides a caching mechanism for function calls (mostly pure functions)

In [31]:
from time import sleep, time

def square(x):
    sleep(1)
    return x * x

values = [2, 2, 5, 2, 6, 3, 6, 2, 2, 3, 3, 5, 6, 6, 2, 3, 5]

start = time()
results = [square(v) for v in values]
end = time()
print(f"Time taken without caching: {end - start} seconds")

Time taken without caching: 17.064401149749756 seconds


In [33]:
from time import sleep, time

def cached(func):
    cache = {}
    def wrapper(*args, **kwargs):
        key = (args, tuple(sorted(kwargs.items())))
        if key not in cache:
            cache[key] = func(*args, **kwargs)
        return cache[key]
    return wrapper

@cached
def square(x):
    sleep(1)
    return x * x

values = [2, 2, 5, 2, 6, 3, 6, 2, 2, 3, 3, 5, 6, 6, 2, 3, 5]

start = time()
results = [square(v) for v in values]
end = time()
print(f"Time taken with caching: {end - start} seconds")

Time taken with caching: 4.009718179702759 seconds


In [38]:
from time import sleep, time

from functools import cache, lru_cache


@lru_cache(maxsize=4)
def square(x):
    print("Square invoked with", x)
    sleep(1)
    return x * x

values = [2, 2, 5, 2, 6, 3, 6, 2, 2, 3, 3, 5, 6, 6, 2, 3, 5, 4, 12, 78, 14, 2]

start = time()
results = [square(v) for v in values]
end = time()
print(f"Time taken with caching: {end - start} seconds")

Square invoked with 2
Square invoked with 5
Square invoked with 6
Square invoked with 3
Square invoked with 4
Square invoked with 12
Square invoked with 78
Square invoked with 14
Square invoked with 2
Time taken with caching: 9.025467872619629 seconds


In [ ]:
import requests

res = requests.get("https://python.org/")
res.status_code
d1 = res.text

In [ ]:
import cached_requests as requests # cached_requests.py should implement caching

res = requests.get("https://python.org/")
res.status_code
d2 = res.text

Fetching URL from cache: https://python.org/


In [ ]:
requests.get("https://chandrashekar.info/")

Fetching URL from cache: https://chandrashekar.info/


<Response [200]>

In [ ]:
requests.requests_cache

{'https://python.org/': <Response [200]>,
 'https://chandrashekar.info/': <Response [200]>}

#### Exercise: Implement a ProxyPath class that also implements a [] operator to access child elements (subfolders and files) within a path, support len() and iterability.


### Descriptor Pattern (Python-specific)

It allows creating rules for attributes using attribute descriptors.
Attribute descriptors are classes with `__get__()` and `__set__()` implemented.

This pattern allows clean separation of logic for applying validation rules for attributes.

In [92]:
class Celsius:  
    store = {}

    def __get__(self, instance, owner=None):
        print("Celsius.__get__ invoked: instance =", instance, ", owner =", owner)
        if instance in self.store:
            return self.store[instance]
        else:
            return 0.0
    
    def __set__(self, instance, value):
        print("Celsius.__set__ invoked: instance =", instance, ", value =", value)
        if value < -273.15:
            raise ValueError("Temperature below -273.15 is not possible.")
        self.store[instance] = value

class Temperature:
    value = Celsius()

temp = Temperature()
print(temp.value)

temp.value = 25
print(temp.value)


Celsius.__get__ invoked: instance = <__main__.Temperature object at 0x1455c4500> , owner = <class '__main__.Temperature'>
0.0
Celsius.__set__ invoked: instance = <__main__.Temperature object at 0x1455c4500> , value = 25
Celsius.__get__ invoked: instance = <__main__.Temperature object at 0x1455c4500> , owner = <class '__main__.Temperature'>
25


In [12]:
# Older approach

class Temperature:
    def __init__(self):
        self._value = 0.0
    
    @property
    def value(self):
        return self._value
    
    @value.setter
    def value(self, new_value):
        if type(new_value) not in (int, float):
            raise TypeError("Temperature value must be a number.")
        if new_value < -273.15:
            raise ValueError("Temperature below -273.15 is not possible.")
        self._value = new_value
    
temp = Temperature()
print(temp.value)
temp.value = 25
print(temp.value)
#temp.value = "hello"

0.0
25


In [102]:
# Newer approach

class Celsius:
    def __init__(self):
        self._values = {}

    def __get__(self, instance, owner=None):
        return self._values.get(instance, 0.0)
    
    def __set__(self, instance, value):
        if type(value) not in (int, float):
            raise TypeError("Temperature value must be a number.")
        if value < -273.15:
            raise ValueError("Temperature below -273.15 is not possible.")
        self._values[instance] = value

class Temperature:
    value = Celsius()

t1 = Temperature()
t2 = Temperature()

t1.value = 10
t2.value = 20
print(t1.value)
print(t2.value)

print(t1.__dict__)
print(t1.__dict__)

10
20
{}
{}


In [106]:
Temperature.__dict__['value']._values

{<__main__.Temperature at 0x1455b53a0>: 10,
 <__main__.Temperature at 0x1455b4950>: 20}

---

### Behavioral Patterns

#### Chain of Responsibility

The Chain of Responsibility pattern passes a request along a chain of handlers, where each handler decides to either process it, skip it, or stop it — without the sender knowing which handler will handle it.

```
Sender → Handler1 → Handler2 → Handler3 → ... → (unhandled)
              ↓          ↓          ↓
           handle      skip       handle
           (stop)    (pass on)   (stop)
```

In [1]:
from __future__ import annotations
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import datetime
from typing import Optional
import time

# ─────────────────────────────────────────
# REQUEST / RESPONSE — the objects travelling the chain
# ─────────────────────────────────────────

@dataclass
class HttpRequest:
    method:   str
    path:     str
    headers:  dict = field(default_factory=dict)
    body:     dict = field(default_factory=dict)
    metadata: dict = field(default_factory=dict)  # handlers annotate this

    def __str__(self):
        return f"{self.method} {self.path}"


@dataclass
class HttpResponse:
    status:  int
    body:    str
    headers: dict = field(default_factory=dict)

    def __str__(self):
        return f"HTTP {self.status}: {self.body}"


# ─────────────────────────────────────────
# HANDLER — abstract base
# ─────────────────────────────────────────

class Middleware(ABC):
    def __init__(self, name: str):
        self.name = name
        self._next: Optional[Middleware] = None

    def set_next(self, handler: Middleware) -> Middleware:
        self._next = handler
        return handler          # allows chaining: a.set_next(b).set_next(c)

    def pass_to_next(self, request: HttpRequest) -> Optional[HttpResponse]:
        if self._next:
            return self._next.handle(request)
        return HttpResponse(500, "No handler could process this request")

    @abstractmethod
    def handle(self, request: HttpRequest) -> Optional[HttpResponse]:
        ...

    def __str__(self):
        return self.name


# ─────────────────────────────────────────
# CONCRETE HANDLERS
# ─────────────────────────────────────────

class AuthMiddleware(Middleware):
    VALID_TOKENS = {"secret-token-123", "admin-token-456"}

    def __init__(self):
        super().__init__("AuthMiddleware")

    def handle(self, request: HttpRequest) -> Optional[HttpResponse]:
        print(f"  [{self.name}] Checking auth...")
        token = request.headers.get("Authorization", "")

        if not token:
            print(f"  [{self.name}] No token — BLOCKED")
            return HttpResponse(401, "Unauthorized: No token provided")

        if token not in self.VALID_TOKENS:
            print(f"  [{self.name}] Invalid token — BLOCKED")
            return HttpResponse(403, "Forbidden: Invalid token")

        # Annotate request with identity for downstream handlers
        request.metadata["user"] = "admin" if "admin" in token else "user"
        print(f"  [{self.name}] Authenticated as '{request.metadata['user']}'")
        return self.pass_to_next(request)


class RateLimitMiddleware(Middleware):
    def __init__(self, max_requests: int, window_seconds: int = 60):
        super().__init__("RateLimitMiddleware")
        self.max_requests    = max_requests
        self.window_seconds  = window_seconds
        self._request_log: dict[str, list[float]] = {}  # user → timestamps

    def handle(self, request: HttpRequest) -> Optional[HttpResponse]:
        print(f"  [{self.name}] Checking rate limit...")
        user = request.metadata.get("user", "anonymous")
        now  = time.time()

        # Purge old timestamps outside the window
        timestamps = self._request_log.get(user, [])
        timestamps = [t for t in timestamps if now - t < self.window_seconds]

        if len(timestamps) >= self.max_requests:
            print(f"  [{self.name}] Rate limit exceeded for '{user}' — BLOCKED")
            return HttpResponse(429, f"Too Many Requests: limit is "
                                     f"{self.max_requests} per {self.window_seconds}s")

        timestamps.append(now)
        self._request_log[user] = timestamps
        remaining = self.max_requests - len(timestamps)
        print(f"  [{self.name}] OK ({remaining} requests remaining)")
        return self.pass_to_next(request)


class ValidationMiddleware(Middleware):
    # Rules: path → list of required body fields
    REQUIRED_FIELDS = {
        "/api/users":   ["username", "email"],
        "/api/payment": ["amount", "card_number", "cvv"],
    }

    def __init__(self):
        super().__init__("ValidationMiddleware")

    def handle(self, request: HttpRequest) -> Optional[HttpResponse]:
        print(f"  [{self.name}] Validating request...")
        required = self.REQUIRED_FIELDS.get(request.path, [])

        if request.method in ("POST", "PUT") and required:
            missing = [f for f in required if f not in request.body]
            if missing:
                print(f"  [{self.name}] Missing fields: {missing} — BLOCKED")
                return HttpResponse(400, f"Bad Request: missing fields {missing}")

        print(f"  [{self.name}] Validation passed")
        return self.pass_to_next(request)


class LoggingMiddleware(Middleware):
    def __init__(self):
        super().__init__("LoggingMiddleware")
        self._log: list[str] = []

    def handle(self, request: HttpRequest) -> Optional[HttpResponse]:
        start   = time.time()
        print(f"  [{self.name}] 📝 Logging request: {request}")

        response = self.pass_to_next(request)   # let chain proceed first

        elapsed = (time.time() - start) * 1000
        entry   = (f"{datetime.now():%H:%M:%S} | {request} | "
                   f"user={request.metadata.get('user','?')} | "
                   f"status={response.status if response else '?'} | "
                   f"{elapsed:.1f}ms")
        self._log.append(entry)
        print(f"  [{self.name}] 📝 Logged: {entry}")
        return response

    def dump_log(self):
        print("\n=== Access Log ===")
        for entry in self._log:
            print(f"  {entry}")


# ─────────────────────────────────────────
# ACTUAL REQUEST HANDLER — end of chain
# ─────────────────────────────────────────

class RequestHandler(Middleware):
    def __init__(self):
        super().__init__("RequestHandler")
        self._routes = {
            ("GET",  "/api/users"):   self._get_users,
            ("POST", "/api/users"):   self._create_user,
            ("POST", "/api/payment"): self._process_payment,
        }

    def handle(self, request: HttpRequest) -> Optional[HttpResponse]:
        print(f"  [{self.name}] 🎯 Handling {request}")
        route = self._routes.get((request.method, request.path))
        if route:
            return route(request)
        return HttpResponse(404, f"Not Found: {request.path}")

    def _get_users(self, req):
        return HttpResponse(200, '{"users": ["Alice", "Bob", "Charlie"]}')

    def _create_user(self, req):
        return HttpResponse(201, f'{{"created": "{req.body["username"]}"}}')

    def _process_payment(self, req):
        return HttpResponse(200, f'{{"charged": "{req.body["amount"]}"}}')


# ─────────────────────────────────────────
# CHAIN BUILDER
# ─────────────────────────────────────────

def build_pipeline() -> tuple[Middleware, LoggingMiddleware]:
    auth       = AuthMiddleware()
    rate_limit = RateLimitMiddleware(max_requests=3, window_seconds=60)
    validation = ValidationMiddleware()
    logger     = LoggingMiddleware()
    handler    = RequestHandler()

    # Build the chain — each returns next, enabling fluent chaining
    auth.set_next(rate_limit).set_next(validation).set_next(logger).set_next(handler)

    return auth, logger      # entry point + logger for log dump

# ───────────────────────────────────────────────────
# CLIENT CODE — sending requests through the pipeline
# ───────────────────────────────────────────────────

pipeline, logger = build_pipeline()

def send(request: HttpRequest):
    print(f"\n{'─'*55}")
    print(f"{request}")
    print(f"{'─'*55}")
    response = pipeline.handle(request)
    print(f"{'─'*55}")
    print(f"Response → {response}\n")

# 1 — No auth token
send(HttpRequest("GET", "/api/users"))

# 2 — Invalid token
send(HttpRequest("GET", "/api/users",
     headers={"Authorization": "bad-token"}))

# 3 — Valid token, missing fields
send(HttpRequest("POST", "/api/users",
     headers={"Authorization": "secret-token-123"},
     body={"username": "alice"}))   # missing 'email'

# 4 — Valid, complete request
send(HttpRequest("POST", "/api/users",
     headers={"Authorization": "secret-token-123"},
     body={"username": "alice", "email": "alice@example.com"}))

# 5, 6, 7 — Hit rate limit (max 3 per minute; user already used 2)
for i in range(3):
    send(HttpRequest("GET", "/api/users",
         headers={"Authorization": "secret-token-123"}))

logger.dump_log()


───────────────────────────────────────────────────────
GET /api/users
───────────────────────────────────────────────────────
  [AuthMiddleware] Checking auth...
  [AuthMiddleware] No token — BLOCKED
───────────────────────────────────────────────────────
Response → HTTP 401: Unauthorized: No token provided


───────────────────────────────────────────────────────
GET /api/users
───────────────────────────────────────────────────────
  [AuthMiddleware] Checking auth...
  [AuthMiddleware] Invalid token — BLOCKED
───────────────────────────────────────────────────────
Response → HTTP 403: Forbidden: Invalid token


───────────────────────────────────────────────────────
POST /api/users
───────────────────────────────────────────────────────
  [AuthMiddleware] Checking auth...
  [AuthMiddleware] Authenticated as 'user'
  [RateLimitMiddleware] Checking rate limit...
  [RateLimitMiddleware] OK (2 requests remaining)
  [ValidationMiddleware] Validating request...
  [ValidationMiddleware] M

#### Simpler example - pythonic

In [ ]:
def square(x):
    print("Calculating square of", x)
    return x * x

def must_be_integer(x):
    if type(x) is not int:
        raise TypeError("Value must be an integer")
    return x

def must_be_positive(x):
    if x <= 0:
        raise ValueError("Value must be positive")
    return x

def must_be_less_than_100(x):
    if x >= 100:
        raise ValueError("Value must be less than 100")
    return x

# This is also a Strategy pattern — we can swap out different sets of validators
validators = [must_be_integer, must_be_positive, must_be_less_than_100]
def validate_and_compute(fn, x, validators):
    for validator in validators:
        try:
            x = validator(x)   # may raise if invalid
        except (TypeError, ValueError) as e:
            print(f"Validation error: {e}")
            break
    else:
        return square(x)

r = validate_and_compute(square, 10, validators)   # works
print("Result:", r)
print("-" * 40)

r = validate_and_compute(square, -5, validators)   # raises ValueError
print("-" * 40)

r = validate_and_compute(square, 150, validators)  # raises ValueError
print("-" * 40)

Calculating square of 10
Result: 100
----------------------------------------
Validation error: Value must be positive
----------------------------------------
Validation error: Value must be less than 100
----------------------------------------


##### Chain-of-Responsibility implementations in the Python core and standard libraries

1. Module imports (py_module, namespace, ext_module, frozen, builtin)

2. Exception propogation: program exceptions -> threading.excepthook -> sys.excepthook -> core exception handler

3. logging module: supports multi-handler propogation of logs.

In [ ]:
#
# Each handler in logger.handlers[] gets the record in turn
# A handler's level acts as its "can I handle this?" check

multi_logger = logging.getLogger("multi")
multi_logger.setLevel(logging.DEBUG)
multi_logger.propagate = False

debug_only = logging.StreamHandler()
debug_only.setLevel(logging.DEBUG)
debug_only.addFilter(lambda r: r.levelno == logging.DEBUG)  # only DEBUG
debug_only.setFormatter(logging.Formatter("%(message)s"))

error_only = logging.StreamHandler()
error_only.setLevel(logging.ERROR)                           # only ERROR+
error_only.setFormatter(logging.Formatter("%(message)s"))

multi_logger.addHandler(debug_only)
multi_logger.addHandler(error_only)

multi_logger.debug("Checking values...")  # Checking values...
multi_logger.info("All good")             # (neither handler takes it)
multi_logger.error("Disk full!")          # Disk full!

---

### Command Pattern

The Command pattern encapsulates a request as an object, letting you parameterize actions, queue them, support undo/redo, and log operations — all without the caller knowing what the action actually does.

In [9]:
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional

# ─────────────────────────────────────────
# RECEIVER — does the actual work
# ─────────────────────────────────────────

class TextEditor:
    def __init__(self):
        self.text    = ""
        self.bold    = False
        self.italic  = False

    def insert(self, text: str, position: int):
        self.text = self.text[:position] + text + self.text[position:]

    def delete(self, position: int, length: int):
        self.text = self.text[:position] + self.text[position + length:]

    def set_bold(self, value: bool):
        self.bold = value

    def set_italic(self, value: bool):
        self.italic = value

    def __str__(self):
        flags = []
        if self.bold:   flags.append("BOLD")
        if self.italic: flags.append("ITALIC")
        style = f" [{', '.join(flags)}]" if flags else ""
        return f'"{self.text}"{style}'


# ─────────────────────────────────────────
# COMMAND — abstract base
# ─────────────────────────────────────────

class Command(ABC):
    @abstractmethod
    def execute(self): ...

    @abstractmethod
    def undo(self): ...


# ─────────────────────────────────────────
# CONCRETE COMMANDS
# ─────────────────────────────────────────

class InsertTextCommand(Command):
    def __init__(self, editor: TextEditor, text: str, position: int):
        self.editor   = editor
        self.text     = text
        self.position = position

    def execute(self):
        self.editor.insert(self.text, self.position)

    def undo(self):
        self.editor.delete(self.position, len(self.text))

    def __str__(self):
        return f"Insert('{self.text}' @ {self.position})"


class DeleteTextCommand(Command):
    def __init__(self, editor: TextEditor, position: int, length: int):
        self.editor   = editor
        self.position = position
        self.length   = length
        self._deleted = ""              # captured on execute for undo

    def execute(self):
        self._deleted = self.editor.text[self.position:self.position + self.length]
        self.editor.delete(self.position, self.length)

    def undo(self):
        self.editor.insert(self._deleted, self.position)

    def __str__(self):
        return f"Delete({self.length} chars @ {self.position})"


class BoldCommand(Command):
    def __init__(self, editor: TextEditor):
        self.editor    = editor
        self._previous = False

    def execute(self):
        self._previous = self.editor.bold
        self.editor.set_bold(True)

    def undo(self):
        self.editor.set_bold(self._previous)

    def __str__(self):
        return "Bold(ON)"


class ItalicCommand(Command):
    def __init__(self, editor: TextEditor):
        self.editor    = editor
        self._previous = False

    def execute(self):
        self._previous = self.editor.italic
        self.editor.set_italic(True)

    def undo(self):
        self.editor.set_italic(self._previous)

    def __str__(self):
        return "Italic(ON)"


# ─────────────────────────────────────────
# MACRO COMMAND — composite of commands
# ─────────────────────────────────────────

class MacroCommand(Command):
    """Executes multiple commands as one atomic unit."""
    def __init__(self, name: str, commands: list[Command]):
        self.name     = name
        self.commands = commands

    def execute(self):
        for cmd in self.commands:
            cmd.execute()

    def undo(self):
        for cmd in reversed(self.commands):   # undo in reverse order
            cmd.undo()

    def __str__(self):
        return f"Macro({self.name})"


# ─────────────────────────────────────────
# INVOKER — manages history, undo, redo
# ─────────────────────────────────────────

class EditorController:
    def __init__(self):
        self._history: list[Command] = []
        self._redos:   list[Command] = []

    def execute(self, command: Command):
        command.execute()
        self._history.append(command)
        self._redos.clear()             # new action clears redo stack
        print(f"  {command}")

    def undo(self):
        if not self._history:
            print("  Nothing to undo")
            return
        command = self._history.pop()
        command.undo()
        self._redos.append(command)
        print(f"  Undid: {command}")

    def redo(self):
        if not self._redos:
            print("  Nothing to redo")
            return
        command = self._redos.pop()
        command.execute()
        self._history.append(command)
        print(f"  Redid: {command}")

    def history(self):
        return [str(c) for c in self._history]

In [10]:
editor     = TextEditor()
controller = EditorController()

print("=== Typing ===")
controller.execute(InsertTextCommand(editor, "Hello",  0))  ; print(f"  → {editor}")
controller.execute(InsertTextCommand(editor, " World", 5))  ; print(f"  → {editor}")
controller.execute(BoldCommand(editor))                     ; print(f"  → {editor}")
controller.execute(ItalicCommand(editor))                   ; print(f"  → {editor}")

print("\n=== Undo x2 ===")
controller.undo() ; print(f"  → {editor}")
controller.undo() ; print(f"  → {editor}")

print("\n=== Redo x1 ===")
controller.redo() ; print(f"  → {editor}")

print("\n=== Delete ===")
controller.execute(DeleteTextCommand(editor, 5, 6))  ; print(f"  → {editor}")

print("\n=== Macro: bold italic insert ===")
macro = MacroCommand("Heading", [
    BoldCommand(editor),
    ItalicCommand(editor),
    InsertTextCommand(editor, "!", 5),
])
controller.execute(macro) ; print(f"  → {editor}")

print("\n=== Undo macro (atomic) ===")
controller.undo()         ; print(f"  → {editor}")

print("\n=== History ===")
for i, cmd in enumerate(controller.history(), 1):
    print(f"  {i}. {cmd}")

=== Typing ===
  Insert('Hello' @ 0)
  → "Hello"
  Insert(' World' @ 5)
  → "Hello World"
  Bold(ON)
  → "Hello World" [BOLD]
  Italic(ON)
  → "Hello World" [BOLD, ITALIC]

=== Undo x2 ===
  Undid: Italic(ON)
  → "Hello World" [BOLD]
  Undid: Bold(ON)
  → "Hello World"

=== Redo x1 ===
  Redid: Bold(ON)
  → "Hello World" [BOLD]

=== Delete ===
  Delete(6 chars @ 5)
  → "Hello" [BOLD]

=== Macro: bold italic insert ===
  Macro(Heading)
  → "Hello!" [BOLD, ITALIC]

=== Undo macro (atomic) ===
  Undid: Macro(Heading)
  → "Hello" [BOLD]

=== History ===
  1. Insert('Hello' @ 0)
  2. Insert(' World' @ 5)
  3. Bold(ON)
  4. Delete(6 chars @ 5)


---

#### A Simpler Pythonic implementation of Command Pattern


In [2]:
from typing import Callable, List, Tuple, Optional
from dataclasses import dataclass
from enum import Enum
from functools import partial

# --- Devices (Receivers) ---
class Light:
    def __init__(self, name: str):
        self.name = name
        self.is_on = False
        self.brightness = 50
    
    def on(self):
        self.is_on = True
        print(f"   {self.name} light turned ON")
    
    def off(self):
        self.is_on = False
        print(f"   {self.name} light turned OFF")
    
    def set_brightness(self, level: int):
        self.brightness = level
        print(f"   {self.name} brightness set to {level}%")

class Thermostat:
    def __init__(self):
        self.temperature = 21
    
    def set_temperature(self, temp: int):
        self.temperature = temp
        print(f"   Thermostat set to {temp}°C")


class SimpleRemote:
    def __init__(self):
        self._commands = {}
    
    def register_button(self, button: str, action: Callable):
        """Register any callable to a button"""
        self._commands[button] = action
    
    def press(self, button: str):
        """Press a button to execute its action"""
        if button in self._commands:
            print(f"🔘 Pressed '{button}' button")
            self._commands[button]()
        else:
            print(f" No action assigned to '{button}'")

# Usage
light = Light("Bedroom")
fan = Thermostat()  # Reusing Thermostat as a mock fan

remote = SimpleRemote()

# Register commands (using functions, lambdas, or partials)
remote.register_button("A", light.on)
remote.register_button("B", light.off)
remote.register_button("C", lambda: light.set_brightness(100))
remote.register_button("D", partial(light.set_brightness, 0))
remote.register_button("E", lambda: fan.set_temperature(18))

# Use the remote
remote.press("A")  # Turns light on
remote.press("C")  # Sets brightness to 100%
remote.press("E")  # Sets fan temperature to 18
remote.press("Z")  # Invalid button

🔘 Pressed 'A' button
   Bedroom light turned ON
🔘 Pressed 'C' button
   Bedroom brightness set to 100%
🔘 Pressed 'E' button
   Thermostat set to 18°C
 No action assigned to 'Z'


---

#### Mediator Pattern

The Mediator Pattern reduces chaotic dependencies between objects by making them communicate through a central mediator instead of directly with each other. Instead of objects referring to each other directly, they only know the mediator.

In [ ]:
from abc import ABC, abstractmethod
from typing import List, Optional
from datetime import datetime

# --- Mediator Interface ---
class ChatMediator(ABC):
    @abstractmethod
    def send_message(self, message: str, sender: 'User'):
        pass
    
    @abstractmethod
    def add_user(self, user: 'User'):
        pass
    
    @abstractmethod
    def remove_user(self, user: 'User'):
        pass

# --- Concrete Mediator ---
class ChatRoom(ChatMediator):
    def __init__(self, name: str):
        self.name = name
        self.users: List['User'] = []
        self.message_history: List[dict] = []
    
    def add_user(self, user: 'User'):
        self.users.append(user)
        self._broadcast_system_message(f"✨ {user.name} joined the chat", user)
    
    def remove_user(self, user: 'User'):
        if user in self.users:
            self.users.remove(user)
            self._broadcast_system_message(f"👋 {user.name} left the chat", user)
    
    def send_message(self, message: str, sender: 'User'):
        timestamp = datetime.now().strftime("%H:%M:%S")
        
        # Log message
        self.message_history.append({
            'sender': sender.name,
            'message': message,
            'timestamp': timestamp
        })
        
        # Broadcast to all users except sender
        for user in self.users:
            if user != sender:
                user.receive_message(message, sender.name, timestamp)
    
    def _broadcast_system_message(self, system_message: str, excluded_user: Optional['User'] = None):
        for user in self.users:
            if user != excluded_user:
                user.receive_system_message(system_message)
    
    def get_chat_history(self, requester: 'User') -> List[dict]:
        """Allow users to see chat history"""
        return self.message_history[-20:]  # Last 20 messages

# --- Colleague ---
class User:
    def __init__(self, name: str, mediator: ChatMediator):
        self.name = name
        self.mediator = mediator
        self.mediator.add_user(self)
    
    def send(self, message: str):
        print(f"\n💬 {self.name}: {message}")
        self.mediator.send_message(message, self)
    
    def receive_message(self, message: str, sender: str, timestamp: str):
        print(f"[{timestamp}] {sender} -> {self.name}: {message}")
    
    def receive_system_message(self, message: str):
        print(f"📢 System -> {self.name}: {message}")
    
    def leave(self):
        self.mediator.remove_user(self)
    
    def view_history(self):
        print(f"\n📜 {self.name}'s chat history:")
        for msg in self.mediator.get_chat_history(self):
            print(f"  [{msg['timestamp']}] {msg['sender']}: {msg['message']}")

# Usage
chat_room = ChatRoom("Python Developers")

alice = User("Alice", chat_room)
bob = User("Bob", chat_room)
charlie = User("Charlie", chat_room)

alice.send("Hey everyone! 👋")
bob.send("Hi Alice! How's it going?")
charlie.send("Hello from Charlie!")
bob.send("Anyone up for some coding?")
charlie.leave()
alice.send("Where did Charlie go?")
bob.view_history()

---

#### Strategy Pattern


In [ ]:
class Parser:
    registry = {}

    def __new__(cls, source):   
        ext = source.split('.')[-1]
        parser_class = cls.get_parser(ext)
        instance = super().__new__(parser_class)
        return instance

    @classmethod
    def register(cls, format_name):
        def decorator(subclass):
            cls.registry[format_name] = subclass
            return subclass
        return decorator

    @classmethod
    def get_parser(cls, format_name):
        parser_class = cls.registry.get(format_name)
        if not parser_class:
            raise ValueError(f"No parser registered for format: {format_name}")
        return parser_class

    def __init__(self, source):
        self.source = source
    
    def parse(self):
        print(f"Parsing source: {self.source}")

#--------------------------------------------
@Parser.register('json')  # Explicit is better than implicit
class JSONParser(Parser):
    def parse(self):
        print(f"Parsing JSON source: {self.source}")

@Parser.register('xml')
class XMLParser(Parser):
    def parse(self):
        print(f"Parsing XML source: {self.source}")

@Parser.register('csv')
class CSVParser(Parser):
    def parse(self):
        print(f"Parsing CSV source: {self.source}")

#-----------------------------------------

p1 = Parser("testfile.json") # Return an instance of JSONParser
p2 = Parser("testfile.xml") # Return an instance of XMLParser
p1.parse()
p2.parse()

In [ ]:
class Parser:

    @classmethod
    def register(cls, format_name):
        def decorator(subclass):
            setattr(cls, format_name, subclass)
            #cls.__dict__[format_name] = subclass
            return subclass
        return decorator

    def __init__(self, source):
        self.source = source
    
    def parse(self):
        print(f"Parsing source: {self.source}")

#--------------------------------------------
@Parser.register('json')
class JSONParser(Parser):
    def parse(self):
        print(f"Parsing JSON source: {self.source}")

@Parser.register('xml')
class XMLParser(Parser):
    def parse(self):
        print(f"Parsing XML source: {self.source}")

@Parser.register('csv')
class CSVParser(Parser):
    def parse(self):
        print(f"Parsing CSV source: {self.source}")

#-----------------------------------------

filename = "testdata.json"
ext = filename.split('.')[-1]


json_parser = getattr(Parser, ext)('data.json')
print(json_parser)
